In [16]:
import os
import rasterio
import tifffile
from rasterio.enums import Resampling

In [17]:
def divide_raster_into_patches(input_file, output_dir, patch_size = 256):
    # Open the input raster file
    with rasterio.open(input_file) as src:
        num_cols = src.width
        num_rows = src.height
        num_bands = src.count

        # Calculate number of patches in each dimension
        num_patches_x = num_cols // patch_size
        num_patches_y = num_rows // patch_size

        # Iterate over each patch
        for i in range(num_patches_x):
            for j in range(num_patches_y):
                # Calculate the window for this patch
                window = rasterio.windows.Window(i * patch_size, j * patch_size, patch_size, patch_size)

                # Read the patch data
                patch = src.read(window=window)

                # Define metadata for the patch
                meta = src.meta.copy()
                meta['width'], meta['height'] = patch_size, patch_size
                meta['transform'] = rasterio.windows.transform(window, src.transform)

                # Create output file path
                output_path = os.path.join(output_dir, f"PAN_U16_{i}_{j}.tif")

                # Write the patch to a new raster file
                with rasterio.open(output_path, 'w', **meta) as dst:
                    dst.write(patch)


In [18]:
input_file = "D:/geoinformatics/MajorProject/Results/tif_results/uint16/WG_PAN_CLIP_BLOCK.TIF"   # Path to your input raster file
output_dir = "D:/geoinformatics/MajorProject/Results/tif_results/uint16/patches/uint16PAN" # Directory where patches will be saved

In [9]:
import numpy as np

__changing datatype__

In [11]:
input_file = "D:/geoinformatics/MajorProject/Results/tif_results/uint32/WG_PAN_CLIP_BLOCK.TIF"
output_file = "D:/geoinformatics/MajorProject/Results/tif_results/uint16/WG_PAN_CLIP_BLOCK.TIF"

In [15]:
# Open the input file
with rasterio.open(input_file, 'r') as src:
    # Read the data
    data_uint32 = src.read()

    # Convert the data to uint16
    data_uint16 = data_uint32.astype('uint16')

    # Get the metadata and update the data type
    metadata = src.meta
    metadata.update(dtype='uint16')

    # Check the nodata value and update it if necessary
    nodata = metadata.get('nodata', None)
    if nodata is not None:
        if nodata > 65535 or nodata < 0:
            # Set nodata to a valid value for uint16, e.g., 0 or 65535
            metadata['nodata'] = 65535

    # Write the converted data to the output file
    with rasterio.open(output_file, 'w', **metadata) as dst:
        dst.write(data_uint16)

print(f"Converted image saved to {output_file}")

Converted image saved to D:/geoinformatics/MajorProject/Results/tif_results/uint16/WG_PAN_CLIP_BLOCK.TIF


In [20]:
os.makedirs(output_dir, exist_ok=True)
divide_raster_into_patches(input_file, output_dir)